In [94]:
import torch
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else "cpu"

In [95]:
docs = [
    "My love is like red, red roses.",
    "Roses are red, violets are blue.",
    "Moses supposes his toes-es are roses."
]

In [96]:
doc_size = len(docs)

In [97]:
vocab = sorted(list(set(word.lower() for doc in docs for word in doc.split())))
vocab_size = len(vocab)


#create a mapping from characters to integers
stoi = {s: i for i, s in enumerate(vocab)}
itos = {i : s for s, i in stoi.items()}
encode = lambda s : [stoi[c] for c in s] #encoder: take a sting, output a list of integers
decode = lambda l: ' '.join([itos[i] for i in l]) #decoder: take a list of integers, output a string

In [102]:
def create_bow(docs):
    bow = torch.zeros(vocab_size, doc_size, dtype=torch.float32, device=device)

    for idx , doc in enumerate(docs):
        for word in doc.split():
            bow[stoi[word.lower()], idx] += 1

    return bow

bow = create_bow(docs=docs)

In [104]:
bow

tensor([[0., 2., 1.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 1., 0.],
        [0., 1., 0.],
        [1., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.]])

In [105]:
def svd_k(bow, k = 2):
    col_space, S, row_space = torch.linalg.svd(bow, full_matrices=False)
    U = col_space[:, :k]
    S = torch.diag(S[:k])
    V_t = row_space[:k, :]

    return U, S, V_t 

U_k, S_k, V_k = svd_k(bow=bow)

In [106]:
print(U_k.shape, S_k.shape, V_k.shape)

torch.Size([15, 2]) torch.Size([2, 2]) torch.Size([2, 3])


In [108]:
bow.shape, (U_k @ S_k @ V_k).shape

(torch.Size([15, 3]), torch.Size([15, 3]))

In [107]:
torch.all(bow == (U_k @ S_k @ V_k))

tensor(False)

In [110]:
def query_lsi(query):
    tokens = query.split()
    q_vec = torch.zeros(vocab_size, dtype=torch.float32)
    for w in tokens:
        q_vec[stoi[w]] += 1
    
    # this query is now vocab space (column space)
    # we need to project it docs space (row space)
    S_k_inv = torch.inverse(S_k)

    lsi_vec = q_vec.T @ U_k @ S_k_inv

    print(lsi_vec.shape, V_k.shape)


    # we will do cosine similarity
    similarities = F.cosine_similarity(lsi_vec.unsqueeze(0), V_k.T, dim=1)

    indices = torch.argsort(similarities, descending=True)

    return [(docs[idx], similarities[idx].item()) for idx in indices]
    

query_lsi("red roses")

torch.Size([2]) torch.Size([2, 3])


[('My love is like red, red roses.', 0.8209282755851746),
 ('Moses supposes his toes-es are roses.', 0.7893930673599243),
 ('Roses are red, violets are blue.', 0.4922110140323639)]